In [ ]:
import json

import cmasher as cmr
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
from scipy.stats import lognorm

# plt.rc("font", family="Nimbus Roman")
plt.rc("font", family="Nimbus Roman No9 L")

SMALL_SIZE = 7
MEDIUM_SIZE = 7
BIGGER_SIZE = 7

plt.rc("font", size=SMALL_SIZE)  # controls default text sizes
plt.rc("axes", titlesize=SMALL_SIZE)  # fontsize of the axes title
plt.rc("axes", labelsize=MEDIUM_SIZE)  # fontsize of the x and y labels
plt.rc("xtick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("ytick", labelsize=SMALL_SIZE)  # fontsize of the tick labels
plt.rc("legend", fontsize=SMALL_SIZE)  # legend fontsize
plt.rc("figure", titlesize=BIGGER_SIZE)  # fontsize of the figure title

In [ ]:
sim_dir = "/Users/z5114326/Documents/simulations/"
# sim_dir = "/Volumes/Expansion/simulations/"
sim_lst = ["m12b", "m12c", "m12f", "m12i", "m12m"]

snap = 600

public_snapshot_file = sim_dir + "snapshot_times_public.txt"
pub_data = pd.read_table(public_snapshot_file, comment="#", header=None, sep=r"\s+")
pub_data.columns = [
    "index",
    "scale_factor",
    "redshift",
    "time_Gyr",
    "lookback_time_Gyr",
    "time_width_Myr",
]
timez0 = float(pub_data[pub_data["index"] == snap]["time_Gyr"].values[0])
pub_snaps = np.array(pub_data["index"])
pub_tims = np.array(pub_data["time_Gyr"])

all_data_fil = sim_dir + "/" + "m12i" + "/" + "m12i" + "_res7100/" + "snapshot_times.txt"
all_data = pd.read_table(all_data_fil, comment="#", header=None, sep=r"\s+")
all_data.columns = [
    "index",
    "scale_factor",
    "redshift",
    "time_Gyr",
    "lookback_time_Gyr",
    "time_width_Myr",
]
all_times = np.array(all_data["time_Gyr"])
all_snaps = np.array(all_data["index"])

sim_codes = sim_dir + "simulation_codes.json"
with open(sim_codes) as sim_json:
    sim_data = json.load(sim_json)

scmap = plt.get_cmap("tab10")
colors = [scmap(i) for i in np.linspace(0, 1, len(sim_lst))]
sim_colors = {sim: colors[i] for i, sim in enumerate(sim_lst)}


In [ ]:
# insitu_only = False
# cmap = cmr.tropical

# # fig, axs = plt.subplots(nrows=2, ncols=3, figsize=(16, 8))
# fig, axs = plt.subplots(3, 2, figsize=(3.5, 3.75))
# plt.subplots_adjust(hspace=0.085, wspace=0.085)

# mass_bins = np.arange(4, 7.5, 0.5)
# mass_labs = [f"{mass_bins[i - 1]} - {mass_bins[i]}" for i in range(1, len(mass_bins))]
# colors = cmap(np.linspace(0, 1, len(mass_labs)))

# dage_bins = np.arange(0, 14.5, 0.5)
# # dage_bins = np.arange(0, 15, 1)
# dage_bin_centers = 0.5 * (dage_bins[:-1] + dage_bins[1:])

# for s, ax in enumerate(axs.flatten()):
#     if s > len(sim_lst) - 1:
#         continue
#     sim = sim_lst[s]
#     ghost_file = f"{sim_dir}{sim}/{sim}_ghosts.hdf5"
#     ghost_data = h5py.File(ghost_file, "r")

#     # Accumulators (sum + count for proper averaging)
#     sum_dict = {mass_lab: np.zeros(len(dage_bin_centers)) for mass_lab in mass_labs}
#     count_dict = {mass_lab: np.zeros(len(dage_bin_centers)) for mass_lab in mass_labs}

#     # --- Loop over all snapshots ---
#     for it_id in ghost_data.keys():
#         src_dat = ghost_data[it_id]["source"]

#         smsk = src_dat["s_flag"][()].astype(bool)
#         imsk = np.ones(len(smsk), dtype=bool)
#         if insitu_only:
#             imsk = src_dat["grpid"][()] == 0

#         mfor = src_dat["logm_tfor"][~smsk & imsk]
#         dage = src_dat["tdis"][~smsk & imsk] - src_dat["tfor"][~smsk & imsk]
#         teig = src_dat["tideig.avg"][~smsk & imsk]

#         mass_idx = np.digitize(mfor, mass_bins)
#         dage_idx = np.digitize(dage, dage_bins)

#         for i in range(1, len(mass_bins)):
#             mmask = mass_idx == i
#             if np.any(mmask):
#                 for j in range(1, len(dage_bins)):
#                     amask = dage_idx == j
#                     mask = mmask & amask

#                     if np.any(mask):
#                         vals = teig[mask]
#                         vals = vals[vals > 0]  # keep only positive values
#                         if len(vals) > 0:
#                             sum_dict[mass_labs[i - 1]][j - 1] += np.sum(vals)
#                             count_dict[mass_labs[i - 1]][j - 1] += len(vals)

#     # --- Compute averages ---
#     avg_dict = {}
#     for mass_lab in mass_labs:
#         avg = np.full(len(dage_bin_centers), np.nan)
#         valid = count_dict[mass_lab] > 0
#         avg[valid] = sum_dict[mass_lab][valid] / count_dict[mass_lab][valid]
#         avg_dict[mass_lab] = avg

#     # --- Plot ---
#     for m, mass_lab in enumerate(mass_labs):
#         if np.nansum(avg_dict[mass_lab]) > 0:
#             lab = r"log$_{10}$ M / M$_{\odot}$ = " + mass_lab
#             ax.plot(avg_dict[mass_lab], dage_bin_centers, label=lab, c=colors[m])

#     ax.set_xscale("log")
#     ax.set_xlim(10**2, 5 * 10**6)
#     ax.set_ylim(0, 14)

#     if s in [0, 2, 4]:
#         ax.set_ylabel("Age at Disruption \n (Gyr)")
#     else:
#         ax.set_yticklabels([])

#     if s < 3:
#         ax.set_xticklabels([])
#     else:
#         ax.set_xlabel(r"$\lambda_m\;(\mathrm{Gyr}^{-2})$")

#     ax.text(0.75, 0.85, sim, transform=ax.transAxes, color="k", ha="left")


# axs[2, 1].axis("off")
# handles, labels = axs[0, 0].get_legend_handles_labels()
# axs[2, 1].legend(
#     handles,
#     labels,
#     loc="center",
#     frameon=False,
#     bbox_to_anchor=(0.5, 0),
# )
# # fig.legend(
# #     handles,
# #     labels,
# #     loc="upper center",
# #     bbox_to_anchor=(0.45, 1.08),
# #     ncol=2,
# #     frameon=True,
# # )

# save_name = "../data/figures/disruption/tidal_age_disruption.pdf"
# fig.savefig(save_name, bbox_inches="tight")

In [ ]:
insitu_only = False
cmap = cmr.tropical

fig, axs = plt.subplots(3, 2, figsize=(3.5, 3.75))
plt.subplots_adjust(hspace=0.085, wspace=0.085)

# Mass bins with final bin = M > 10^6 Msun
# mass_labs = [
#     "4.0 - 4.5",
#     "4.5 - 5.0",
#     "5.0 - 5.5",
#     "5.5 - 6.0",
#     "> 6.0",
# ]
mass_labs = [
    r"10$^{4.0}$ $\leq$ M$_{\rm form}$ / M$_{\odot}$ < 10$^{4.5}$",
    r"10$^{4.5}$ $\leq$ M$_{\rm form}$ / M$_{\odot}$ < 10$^{5.0}$",
    r"10$^{5.0}$ $\leq$ M$_{\rm form}$ / M$_{\odot}$ < 10$^{5.5}$",
    r"10$^{5.5}$ $\leq$ M$_{\rm form}$ / M$_{\odot}$ < 10$^{6.0}$",
    r"10$^{6.0}$ $\leq$ M$_{\rm form}$ / M$_{\odot}$",
]

colors = cmap(np.linspace(0, 1, len(mass_labs)))

dage_bins = np.arange(0, 14.5, 0.5)
dage_bin_centers = 0.5 * (dage_bins[:-1] + dage_bins[1:])

for s, ax in enumerate(axs.flatten()):
    if s > len(sim_lst) - 1:
        continue

    sim = sim_lst[s]
    ghost_file = f"{sim_dir}{sim}/{sim}_ghosts.hdf5"
    ghost_data = h5py.File(ghost_file, "r")

    # Accumulators
    sum_dict = {mass_lab: np.zeros(len(dage_bin_centers)) for mass_lab in mass_labs}

    count_dict = {mass_lab: np.zeros(len(dage_bin_centers)) for mass_lab in mass_labs}

    # Loop over snapshots
    for it_id in ghost_data.keys():
        src_dat = ghost_data[it_id]["source"]

        smsk = src_dat["s_flag"][()].astype(bool)

        imsk = np.ones(len(smsk), dtype=bool)
        if insitu_only:
            imsk = src_dat["grpid"][()] == 0

        mfor = src_dat["logm_tfor"][~smsk & imsk]
        dage = src_dat["tdis"][~smsk & imsk] - src_dat["tfor"][~smsk & imsk]
        teig = src_dat["tideig.avg"][~smsk & imsk]

        # Custom mass bins
        mass_idx = np.full(len(mfor), -1)

        mass_idx[(mfor >= 4.0) & (mfor < 4.5)] = 0
        mass_idx[(mfor >= 4.5) & (mfor < 5.0)] = 1
        mass_idx[(mfor >= 5.0) & (mfor < 5.5)] = 2
        mass_idx[(mfor >= 5.5) & (mfor < 6.0)] = 3
        mass_idx[mfor >= 6.0] = 4

        dage_idx = np.digitize(dage, dage_bins)

        for i, mass_lab in enumerate(mass_labs):
            mmask = mass_idx == i

            if np.any(mmask):
                for j in range(1, len(dage_bins)):
                    amask = dage_idx == j
                    mask = mmask & amask

                    if np.any(mask):
                        vals = teig[mask]
                        vals = vals[vals > 0]

                        if len(vals) > 0:
                            sum_dict[mass_lab][j - 1] += np.sum(vals)
                            count_dict[mass_lab][j - 1] += len(vals)

    # Compute averages
    avg_dict = {}

    for mass_lab in mass_labs:
        avg = np.full(len(dage_bin_centers), np.nan)

        valid = count_dict[mass_lab] > 0
        avg[valid] = sum_dict[mass_lab][valid] / count_dict[mass_lab][valid]

        avg_dict[mass_lab] = avg

    # Plot
    for m, mass_lab in enumerate(mass_labs):
        if np.nansum(avg_dict[mass_lab]) > 0:
            if mass_lab.startswith(">"):
                # lab = r"log$_{10}$ M / M$_{\odot}$ " + mass_lab
                lab = mass_lab
            else:
                # lab = r"log$_{10}$ M / M$_{\odot}$ = " + mass_lab
                lab = mass_lab

            ax.plot(
                avg_dict[mass_lab],
                dage_bin_centers,
                label=lab,
                c=colors[m],
            )

    ax.set_xscale("log")
    ax.set_xlim(10**2, 5 * 10**6)
    ax.set_ylim(0, 14)

    ax.set_xticks([10**2, 10**3, 10**4, 10**5, 10**6])

    if s in [0, 2, 4]:
        ax.set_ylabel("Age at Disruption\n(Gyr)")
    else:
        ax.set_yticklabels([])

    if s < 3:
        ax.set_xticklabels([])
    else:
        ax.set_xlabel(r"$\lambda_m\;(\mathrm{Gyr}^{-2})$")

    ax.text(
        0.75,
        0.85,
        sim,
        transform=ax.transAxes,
        color="k",
        ha="left",
    )

axs[2, 1].axis("off")

handles, labels = axs[0, 0].get_legend_handles_labels()

axs[2, 1].legend(
    handles,
    labels,
    loc="center",
    frameon=False,
    bbox_to_anchor=(0.55, 0.05),
)

# fig.legend(
#     handles,
#     labels,
#     loc="center",
#     frameon=False,
#     bbox_to_anchor=(0.5, 1.0),
#     ncols = 2
# )

save_name = "../data/figures/disruption/tidal_age_disruption.pdf"
fig.savefig(save_name, bbox_inches="tight")

In [ ]:
bo_path = "../data/sim_data/boundary_onset_lbt.json"
with open(bo_path, "r", encoding="utf-8") as f:
    bo_dict = json.load(f)

em_path = "../data/sim_data/early_merger_lbt.json"
with open(em_path, "r", encoding="utf-8") as f:
    em_dict = json.load(f)

In [ ]:
def ln_pdf(x, sigma, scale):
    return lognorm.pdf(x, sigma, loc=0, scale=scale)


age_bins = np.arange(0, 14.5, 0.5)
bin_widths = age_bins[1:] - age_bins[:-1]
bin_centres = 0.5 * (age_bins[:-1] + age_bins[1:])

hist_dict = {sim: {"in_c": None, "in_d": None, "ex_c": None, "ex_d": None} for sim in sim_lst}

for sim in sim_lst:
    ghost_file = f"{sim_dir}{sim}/{sim}_ghosts.hdf5"
    ghost_data = h5py.File(ghost_file, "r")

    insum = None
    exsum = None
    n = 0

    for it_id in ghost_data.keys():
        src_dat = ghost_data[it_id]["source"]

        tfors = src_dat["tfor"][()]
        tdiss = src_dat["tdis"][()]

        imsk = src_dat["grpid"][()] == 0
        smsk = tdiss != -1

        dage_i = tdiss[imsk & smsk] - tfors[imsk & smsk]
        dage_e = tdiss[~imsk & smsk] - tfors[~imsk & smsk]

        hin, _ = np.histogram(dage_i, bins=age_bins)
        hex, _ = np.histogram(dage_e, bins=age_bins)

        if insum is None:
            insum = hin.astype(float)
            exsum = hex.astype(float)
        else:
            insum += hin
            exsum += hex
        n += 1

    hist_dict[sim]["in_c"] = insum / n
    hist_dict[sim]["ex_c"] = exsum / n

    hist_dict[sim]["in_d"] = hist_dict[sim]["in_c"] / np.sum(hist_dict[sim]["in_c"] * bin_widths)
    hist_dict[sim]["ex_d"] = hist_dict[sim]["ex_c"] / np.sum(hist_dict[sim]["ex_c"] * bin_widths)

In [ ]:
xx = np.linspace(0.01, 13.5, 1000)

fig, axs = plt.subplots(nrows=5, ncols=2, figsize=(3.5, 8))
plt.subplots_adjust(hspace=0.1, wspace=0.085)

for i, ax in enumerate(axs.flatten()):
    sim = sim_lst[int(i / 2)]
    if i % 2 == 0:
        x = hist_dict[sim]["in_d"]
        c = "r"
        ax.text(0.75, 0.85, sim, transform=ax.transAxes, color="k", ha="left")

        ax.set_ylabel("Density")

    else:
        x = hist_dict[sim]["ex_d"]
        c = "b"

        ax.set_ylabel("")
        ax.set_yticklabels([])

    mask = x > 0  # avoid empty bins
    popt, pcov = curve_fit(ln_pdf, bin_centres[mask], x[mask], p0=(0.5, 1.5))
    sigma, scale = popt

    # mean, var = lognorm.stats(s=sigma, scale=scale, moments="mv")
    # std = np.sqrt(var)

    ax.step(age_bins[:-1], x, where="post", color=c, lw=0.5)
    ax.fill_between(age_bins[:-1], x, step="post", alpha=0.2, color=c)

    ax.plot(xx, ln_pdf(xx, sigma, scale), "k", lw=1)

    # ax.axvline(mean, c="grey", ls="-", label=r"$\langle$ t$_\mathrm{dis}$ $\rangle$")
    # ax.axvline(mean + std, c="grey", ls="--", label=r"mean + 1$\sigma$")

    t90 = lognorm.ppf(0.90, sigma, scale=scale)
    ax.axvline(t90, ls="--", c="grey", label=r"90% Disrupted")

    if i % 2 == 0:
        tdif = em_dict[sim] - bo_dict["in_situ"][sim]

        frac_lt2 = 100 * lognorm.cdf(tdif, sigma, scale=scale)
        print(f"{sim:10s} t90 = {t90:.2f} Gyr, fraction < {tdif:.1f} Gyr = {frac_lt2:.1f}%")

        ax.axvline(tdif, ls=":", lw=1, c="k")

    # print(sim, np.round(t90, 2))

    ax.set_xlim(0, 9.5)
    ax.set_ylim(0, 0.65)

    if i in [8, 9]:
        ax.set_xlabel("Age at Disruption (Gyr)")
    else:
        ax.set_xticklabels([])

    # if i == 0:
    #     ax.legend(loc="lower right")


save_name = "../data/figures/disruption/age_disruption.pdf"
fig.savefig(save_name, bbox_inches="tight")

In [ ]:
9.30 - 6.48